Code to show distribution of neutrino interaction distribution with reference of neutrino vertex

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

In [2]:
from microboone_utils import *

In [3]:
from file import File

In [4]:
f = File("/nevis/westside/data/sc5303/Data/ubOpen/h5/bnb_WithWire_00.h5")

startevt = 0
nevts = 10 #anything from 0 to len(f)

tables = ['hit_table','edep_table','particle_table','event_table']
for t in tables: f.add_group(t)
f.read_data(startevt, nevts)
evts = f.build_evt()

In [15]:
myevt = 2

evtevts = evts[myevt]['event_table']
evthits = evts[myevt]['hit_table']
evtdeps = evts[myevt]['edep_table']
evtparts = evts[myevt]['particle_table']

In [16]:
evtdeps = evtdeps.sort_values(by=['energy_fraction'], ascending=False, kind='mergesort').drop_duplicates(["hit_id"])

In [17]:
evthits = evthits.merge(evtdeps, on=["hit_id"], how="left")
evthits['g4_id'] = evthits['g4_id'].fillna(-1)
evthits = evthits.fillna(0)

evthits["cosmic_label"] = 'neutrino'
evthits.loc[evthits["g4_id"]<0,"cosmic_label"] = 'cosmic'

In [18]:
evthits = evthits.merge(evtparts[['g4_id','category','instance']], on="g4_id", how="left")

Now make the plot. In order to better visualize the neutrino information, click on the "cosmic" entry in the legend to mask it out. Click and drag on the plot to select a region to zoom in. Mouse over hits to visualize their properties.

In [19]:
evthits["category_name"] = 'cosmic'

for l in category:
    #print(l.name,l.value)
    evthits.loc[evthits["category"]==l.value, "category_name"] = l.name

color_dict = {"pion" : "yellow",
              "muon" : "green",
              "kaon" : "black",
              "proton" : "blue",
              "electron" : "red",
              "michel" : "purple",
              "delta" : "pink",
              "other" : "orange",
              "photon" : "cyan",
              "cosmic": "gray"}

fig = px.scatter(evthits, x="local_wire", y="local_time", color="category_name", color_discrete_map=color_dict, facet_col="local_plane", 
                 labels={
                     "local_wire": "Wire",
                     "local_time": "Time Tick",
                     "category_name": "Class"
                 },
                 hover_data=['integral','instance'],
                 title="semantic_label plot")
fig.update_traces(marker={'size': 4})
fig.show()


In [ ]:
# Filter out 'cosmic' and 'other' from category_name
filtered_hits_plane2 = evthits[evthits['local_plane'] == 2]
filtered = filtered_hits_plane2[~filtered_hits_plane2['category_name'].isin(['cosmic', 'other'])]

# Compute min/max for local_wire and local_time
min_wire = filtered['local_wire'].min()
max_wire = filtered['local_wire'].max()
min_time = filtered['local_time'].min()
max_time = filtered['local_time'].max()

print(f"Wire: min={min_wire}, max={max_wire}")
print(f"Time Tick: min={min_time}, max={max_time}")
print(f"Neutrino Interaction Wire: {evtevts['nu_vtx_wire_pos_2'][0]}, Time: {evtevts['nu_vtx_wire_time'][0]}")



Wire: min=2642, max=2817
Time Tick: min=822.3854370117188, max=1037.7708740234375
Neutrino Interaction Wire: 2700, Time: 3766.342041015625
